#  <u>Earnings Sentiment vs Stock Moves</u>
#
##### **Objective**

- Investigate whether the sentiment expressed in company earnings calls or reports can help explain or predict short-term stock price moves


In [2]:
print("Installing dependencies:")
import numpy as np
import matplotlib as plt
import yfinance as yf
import pandas as pd
from datetime import timedelta
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download("vader_lexicon")

Installing dependencies:


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Mosan\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## Earnings call transcript section

Importing earnings call transcript for Apple inc Q2 2025 - Seeking Alpha

In [3]:
with open("Alphabet_q2_2025.txt", "r", encoding="utf-8") as f:
    text = f.read()


Simple cleaning function and clean


In [4]:
def clean_text(s):
    s = s.lower()                               # lowercase
    s = re.sub(r'\n+', ' ', s)                  # collapse newlines to spaces
    s = re.sub(r'\[.*?\]', '', s)               # remove bracketed annotations [laughter], [inaudible]
    s = re.sub(r'[^a-z0-9\s\.]', '', s)         # remove punctuation except periods and numbers
    s = re.sub(r'\s+', ' ', s).strip()          # collapse multiple spaces and strip ends
    return s

text = clean_text(text)

Running sentiment with VADAR <br>

We could use FinBERT which would be the finance upgrade, however due to GPU issues this project uses VADAR

In [5]:
sia = SentimentIntensityAnalyzer()
sentiment_score = sia.polarity_scores(text)["compound"]  # single score in [-1, 1]
sentiment_score

1.0

## Stock returns section

Get price history for a ticker

In [6]:
ticker = "GOOGL"
earnings_date ="2025-07-23"
t = yf.Ticker(ticker)
hist = t.history(period="1y") # DataFrame with datetime index & OHLCVDivStckSpl
print(hist)
hist.loc[earnings_date] # day before earnings call

                                 Open        High         Low       Close  \
Date                                                                        
2024-09-16 00:00:00-04:00  156.619290  157.555166  155.912416  157.365997   
2024-09-17 00:00:00-04:00  158.321813  159.845094  157.684623  158.620499   
2024-09-18 00:00:00-04:00  159.158106  159.795295  157.903643  159.108322   
2024-09-19 00:00:00-04:00  162.991199  163.070835  160.631595  161.428085   
2024-09-20 00:00:00-04:00  162.782133  163.011119  161.348453  162.871735   
...                               ...         ...         ...         ...   
2025-09-10 00:00:00-04:00  238.899994  241.660004  237.850006  239.169998   
2025-09-11 00:00:00-04:00  239.880005  242.250000  236.250000  240.369995   
2025-09-12 00:00:00-04:00  240.369995  242.080002  238.000000  240.800003   
2025-09-15 00:00:00-04:00  244.660004  252.410004  244.660004  251.610001   
2025-09-16 00:00:00-04:00  252.125000  253.039993  249.470001  251.778198   

Open            1.913289e+02
High            1.923580e+02
Low             1.890109e+02
Close           1.900600e+02
Volume          5.868190e+07
Dividends       0.000000e+00
Stock Splits    0.000000e+00
Name: 2025-07-23 00:00:00-04:00, dtype: float64

In [7]:
def get_price_on_or_before(ticker, date):
    """
    Get the stock's closing price on the event date if it's a trading day,
    otherwise the most recent trading day before it.
    """
    date = pd.to_datetime(date)

    # pull a window of data around the date
    start = date - timedelta(days=10)   # buffer in case of holidays
    end = date + timedelta(days=1)      # just after the date
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)

    if df.empty:
        return None

    # normalize to date only
    trading_days = df.index.normalize()
    target_day = date.normalize()

    # if target_day is in trading_days, use it
    if target_day in trading_days:
        idx = trading_days.get_loc(target_day)
    else:
        # if not, take the most recent trading day BEFORE target_day
        idx_list = (trading_days < target_day).nonzero()[0]
        if len(idx_list) == 0:
            return None
        idx = idx_list[-1]   # last one before target_day

    return {
        "date": df.index[idx].date(),
        "close": df["Close"].iloc[idx].item()   # -> returns a native Python float


    }

In [8]:
price_before = get_price_on_or_before(ticker, earnings_date)  # date of earnings call
print(price_before)

{'date': datetime.date(2025, 7, 23), 'close': 190.0600128173828}


In [9]:
price_nextday = get_price_on_or_before(ticker,"2025-07-24") # one day later
price_nextweek = get_price_on_or_before(ticker,"2025-07-30") # one week later
print(price_nextday)
print(price_nextweek)

{'date': datetime.date(2025, 7, 24), 'close': 191.99827575683594}
{'date': datetime.date(2025, 7, 30), 'close': 196.35438537597656}


dict

In [60]:
# percentage change from before earnings call to next day and next week

ret_next_day = (price_nextday['close'] - price_before['close'])/price_before['close'] *100
ret_next_week = (price_nextweek['close'] - price_before['close'])/price_before['close'] *100
print(ticker,"stock return next day:",ret_next_day,"%")
print(ticker,"stock return next week:",ret_next_week,"%")

GOOGL stock return next day: 1.0198162731449907 %
GOOGL stock return next week: 3.3117816132326756 %


In [63]:
# 7) store results
results = pd.DataFrame([{
    "ticker": ticker,
    "earnings_date": earnings_date,
    "sentiment": sentiment_score,
    "ret_next_day": ret_next_day,
    "ret_next_Cal_week": ret_next_week
}])
print(results)

  ticker earnings_date  sentiment  ret_next_day  ret_next_Cal_week
0  GOOGL    2025-07-23        1.0      1.019816           3.311782
